# pocketHb 03 — train the base regressor

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayanthvee/pocketHb/blob/main/notebooks/03_train.ipynb)

transfer-learn a ResNet18 on the ~720 nail crops (subject-disjoint 70/15/15). target: patient-level test MAE under 1.3 g/dL — meaningfully below the predict-mean floor of 1.77 g/dL from chunk 2.

**training compute:** designed for a free Colab T4 (~3 min). it will run on cpu too (~30 min on small intel), but that's a smoke-test path — production weights come from the gpu run.

**huggingface auth:** at the end, weights get pushed to `bubbaonbubba/pockethb-base`. set `HF_TOKEN` as a colab secret (🔑 icon in left sidebar) before running — see the auth cell below for what to do if you're not me.

In [ ]:
# normalise cwd, colab clone if needed, expose src/
import os, sys, subprocess
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

if not Path('scripts/download_data.py').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/jayanthvee/pocketHb.git'])
    os.chdir('pocketHb')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

sys.path.insert(0, 'src')
print('cwd:', Path.cwd())

In [ ]:
subprocess.check_call([sys.executable, 'scripts/download_data.py'])

In [ ]:
# HF auth — try in order: colab secrets, env var, local HF cache. read-only fallback.
import os
hf_token = None
try:
    from google.colab import userdata  # colab
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass
hf_token = hf_token or os.environ.get('HF_TOKEN')

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('hf: authenticated (write enabled)')
else:
    print('hf: no token found — training will run but weights upload will be skipped')
    print('     (set HF_TOKEN in colab secrets if you want upload)')

In [ ]:
import torch
import numpy as np
import random

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  device:', torch.cuda.get_device_name(0))

## load data + build subject-disjoint splits

In [ ]:
from pockethb.data import load_metadata, subject_disjoint_split
from pockethb.dataset import NailCropDataset, crops_for_patients

df = load_metadata()
splits = subject_disjoint_split(df, ratios=(0.70, 0.15, 0.15), seed=SEED)

train_crops = crops_for_patients(df, splits['train'], region='nail')
val_crops   = crops_for_patients(df, splits['val'],   region='nail')
test_crops  = crops_for_patients(df, splits['test'],  region='nail')

print(f"train crops: {len(train_crops)}")
print(f"val   crops: {len(val_crops)}")
print(f"test  crops: {len(test_crops)}")

train_ds = NailCropDataset(train_crops, train=True)
val_ds   = NailCropDataset(val_crops,   train=False)
test_ds  = NailCropDataset(test_crops,  train=False)

## build model

In [ ]:
from pockethb.model import HbRegressor

model = HbRegressor(backbone='resnet18', pretrained=True)
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'{model.backbone_name}  total params={n_params:,}  trainable={n_train:,}  feature_dim={model.feature_dim}')

## train

In [ ]:
from pockethb.train import TrainConfig, train_model

cfg = TrainConfig(
    epochs=30,
    batch_size=32,
    lr=1e-4,
    weight_decay=1e-4,
    patience=6,
    image_size=224,
    num_workers=2 if torch.cuda.is_available() else 0,
    device='auto',
)

result = train_model(model, train_ds, val_ds, test_ds=test_ds, cfg=cfg)

In [ ]:
# training curve
import matplotlib.pyplot as plt
hist = result.history
epochs = [h['epoch'] for h in hist]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, [h['tr_MAE_p'] for h in hist], label='train MAE (patient)')
ax.plot(epochs, [h['va_MAE_p'] for h in hist], label='val MAE (patient)')
ax.axhline(1.77, color='grey', ls='--', lw=1, label='predict-mean floor (test)')
ax.axvline(result.best_epoch, color='black', ls=':', lw=1, label=f'best epoch={result.best_epoch}')
ax.set_xlabel('epoch'); ax.set_ylabel('patient-level MAE (g/dL)')
ax.set_title('training curve — pocketHb base regressor')
ax.legend(); plt.tight_layout(); plt.show()

## save + push weights to HF Hub

In [ ]:
Path('weights').mkdir(exist_ok=True)
weights_path = Path('weights/pockethb_base.pt')
save_blob = {
    'state_dict': result.best_state_dict,
    'config': {
        'backbone': model.backbone_name,
        'image_size': cfg.image_size,
        'imagenet_mean': [0.485, 0.456, 0.406],
        'imagenet_std':  [0.229, 0.224, 0.225],
    },
    'metrics': {
        'val_MAE_patient_best': result.best_val_mae,
        'best_epoch': result.best_epoch,
        'test': result.test_metrics,
    },
}
torch.save(save_blob, weights_path)
print(f'wrote {weights_path}  ({weights_path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
if hf_token:
    from huggingface_hub import HfApi, create_repo
    REPO_ID = 'bubbaonbubba/pockethb-base'
    api = HfApi()
    create_repo(REPO_ID, repo_type='model', exist_ok=True, private=False)
    api.upload_file(
        path_or_fileobj=str(weights_path),
        path_in_repo='pockethb_base.pt',
        repo_id=REPO_ID,
        repo_type='model',
    )
    # minimal model card
    card = (
        f"# pockethb-base\n\n"
        f"base hemoglobin regressor from [pocketHb](https://github.com/jayanthvee/pocketHb). "
        f"resnet18 finetuned on the [nature sci data 2024](https://www.nature.com/articles/s41597-024-03895-9) "
        f"fingernail+hb dataset (n=250 subjects). subject-disjoint 70/15/15 split, seed 42.\n\n"
        f"## test metrics (patient-level)\n\n"
        f"- MAE: {result.test_metrics['patient']['MAE']:.3f} g/dL\n"
        f"- RMSE: {result.test_metrics['patient']['RMSE']:.3f} g/dL\n"
        f"- R²: {result.test_metrics['patient']['R2']:+.3f}\n"
        f"- n = {result.test_metrics['patient']['n']} subjects\n\n"
        f"## not a medical device\n\n"
        f"research replication only. not FDA cleared. do not use in place of a blood test.\n"
    )
    api.upload_file(
        path_or_fileobj=card.encode('utf-8'),
        path_in_repo='README.md',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print(f'pushed weights + README to https://huggingface.co/{REPO_ID}')
else:
    print('skipping HF push (no token)')

## takeaways

- compare patient-level test MAE against the chunk-2 floor of 1.77 g/dL
- best epoch + best val MAE printed above
- if test MAE > 1.3 g/dL, the next move is to try EfficientNet-B0 (smaller, sometimes generalises better on tiny data) or add nail+skin dual-input — but ResNet18 alone usually suffices to clear that bar